# Qwen3.5-9B Full VQA — SSAFY Run-all Pipeline

이 노트북은 `Qwen/Qwen3.5-9B`를 사용해 다음을 처음부터 끝까지 실행합니다.

1. Colab 환경을 충돌 없이 설치
2. 기존과 같은 마지막 10% validation(508개) 구성
3. Qwen3.5-9B zero-shot 전체 평가
4. gold + 고신뢰 dev pseudo-label로 vision+language LoRA 최대 3 epoch 학습 및 best checkpoint 복원
5. fine-tuned 전체 평가와 카테고리별 zero-shot/fine-tuned router 결정
6. test 전체 추론 및 zero-shot / fine-tuned / routed 제출 3개 생성
7. adapter와 결과를 Google Drive에 백업

권장 하드웨어는 **A100 80GB**입니다. 메뉴에서 `런타임 > 런타임 유형 변경 > A100 GPU`를 선택한 뒤 `런타임 > 모두 실행`을 누르세요.

주의: 이 노트북은 새로운 독립 실험입니다. 기존 Qwen3-VL-8B나 27B adapter를 요구하지 않습니다.


## 1. 환경 설치


In [ ]:
# Colab이 transformers/PIL을 자동 import할 수 있으므로, import 여부가 아니라 설치 버전을 검사합니다.
import sys, subprocess
import importlib
import importlib.metadata as metadata

TARGET_VERSIONS = {"transformers": "5.15.1", "peft": "0.20.0", "Pillow": "11.3.0"}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

torchao_version = installed_version("torchao")
if torchao_version is not None:
    assert "torchao" not in sys.modules, (
        f"torchao {torchao_version}가 이미 메모리에 로드되었습니다. "
        "런타임을 다시 시작한 뒤 첫 셀부터 실행하세요."
    )
    print(f"사용하지 않는 충돌 패키지 torchao {torchao_version} 제거 중...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    importlib.invalidate_caches()
assert installed_version("torchao") is None, "torchao 제거에 실패했습니다."

versions_before = {package: installed_version(package) for package in TARGET_VERSIONS}
versions_ready = all(versions_before[package] == version for package, version in TARGET_VERSIONS.items())

if versions_ready:
    print("필수 버전이 이미 설치되어 있어 재설치를 건너뜁니다:", versions_before)
else:
    preloaded = [name for name in ("transformers", "peft", "PIL") if name in sys.modules]
    if preloaded:
        raise RuntimeError(
            "설치 버전은 맞지 않는데 패키지가 이미 메모리에 로드되었습니다. "
            f"loaded={preloaded}, installed={versions_before}. "
            "Colab 메뉴에서 '런타임 > 연결 해제 및 런타임 삭제' 후 다시 연결하여 이 셀부터 실행하세요."
        )
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade",
        "transformers==5.15.1",
        "peft==0.20.0",
        "accelerate>=1.14.0",
        "safetensors>=0.6.0",
        "Pillow==11.3.0",
        "pandas>=2.2",
        "scikit-learn>=1.5",
        "tqdm>=4.66",
    ], check=True)

versions_after = {package: installed_version(package) for package in TARGET_VERSIONS}
assert versions_after == TARGET_VERSIONS, f"패키지 버전 불일치: {versions_after}"
print("환경 준비 완료:", versions_after)


## 2. 공통 설정


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, math, random, re, shutil, zipfile
from collections import Counter
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
Image.MAX_IMAGE_PIXELS = None

assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."
DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", GPU_NAME, f"{GPU_VRAM_GIB:.1f} GiB")
assert GPU_VRAM_GIB >= 70, (
    f"현재 프로필은 A100 80GB 기준입니다. 감지된 VRAM={GPU_VRAM_GIB:.1f} GiB"
)

MODEL_ID = "Qwen/Qwen3.5-9B"
DATA_ROOT = Path("/content")
DATA_ARCHIVE = Path("/content/drive/MyDrive/2026-ssafy-15-2-ai.zip")
IMAGE_SIZE = 512

EPOCHS = 3
TRAIN_BATCH_SIZE = 8
GRAD_ACCUM = 1
INFER_BATCH_SIZE = 8
NUM_WORKERS = 0
TARGET_EFFECTIVE_BATCH = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MIN_DEV_VOTES = 4
PSEUDO_CAP_PER_CATEGORY = 0.75
AUGMENT_HORIZONTAL_FLIP = True
RUN_ZERO_SHOT_VALID = True
RUN_TEST = True
EXPORT_TO_DRIVE = True

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("/content/qwen35_9b_ssafy") / RUN_STAMP
BEST_ADAPTER_DIR = OUTPUT_ROOT / "best_adapter"
ZERO_VALID_PATH = OUTPUT_ROOT / "qwen35_zero_valid.csv"
TUNED_VALID_PATH = OUTPUT_ROOT / "qwen35_tuned_valid.csv"
ROUTED_VALID_PATH = OUTPUT_ROOT / "qwen35_routed_valid.csv"
ZERO_TEST_PATH = OUTPUT_ROOT / "qwen35_zero_test.csv"
TUNED_TEST_PATH = OUTPUT_ROOT / "qwen35_tuned_test.csv"
SUB_ZERO_PATH = OUTPUT_ROOT / "submission_qwen35_zero.csv"
SUB_TUNED_PATH = OUTPUT_ROOT / "submission_qwen35_tuned.csv"
SUB_ROUTED_PATH = OUTPUT_ROOT / "submission_qwen35_routed.csv"
HISTORY_PATH = OUTPUT_ROOT / "training_history.csv"
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/qwen35_9b_ssafy")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LETTERS = ["a", "b", "c", "d"]

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()

def cuda_memory():
    return {
        "allocated_GB": round(torch.cuda.memory_allocated() / 1e9, 2),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1e9, 2),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

assert TRAIN_BATCH_SIZE * GRAD_ACCUM == TARGET_EFFECTIVE_BATCH
print("train micro/effective batch:", TRAIN_BATCH_SIZE, "/", TRAIN_BATCH_SIZE * GRAD_ACCUM)
print("MODEL:", MODEL_ID)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 3. 데이터 준비와 고정 validation


In [ ]:
required = [
    DATA_ROOT / "train.csv", DATA_ROOT / "dev.csv", DATA_ROOT / "test.csv",
    DATA_ROOT / "train", DATA_ROOT / "dev", DATA_ROOT / "test",
]

if not all(path.exists() for path in required):
    from google.colab import drive
    drive.mount("/content/drive")
    assert DATA_ARCHIVE.exists(), f"데이터 ZIP이 없습니다: {DATA_ARCHIVE}"
    print("데이터 압축 해제:", DATA_ARCHIVE)
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall(DATA_ROOT)

missing = [str(path) for path in required if not path.exists()]
assert not missing, f"필수 데이터 누락: {missing}"

train_df = pd.read_csv(DATA_ROOT / "train.csv")
dev_df = pd.read_csv(DATA_ROOT / "dev.csv")
test_df = pd.read_csv(DATA_ROOT / "test.csv")

def categorize(question):
    question = str(question)
    if "몇 개" in question or "개수" in question:
        return "counting"
    if "재질" in question or "소재" in question:
        return "material"
    if "색" in question:
        return "color"
    if "종류" in question:
        return "type"
    return "other"

for dataframe in (train_df, dev_df, test_df):
    dataframe["category"] = dataframe["question"].map(categorize)

assert train_df["answer"].isin(LETTERS).all()
assert train_df["id"].is_unique and dev_df["id"].is_unique and test_df["id"].is_unique

# 기존 모델과 정확히 비교하기 위해 기존과 동일하게 마지막 10%를 validation으로 유지합니다.
split_index = int(len(train_df) * 0.9)
gold_train_df = train_df.iloc[:split_index].copy().reset_index(drop=True)
valid_df = train_df.iloc[split_index:].copy().reset_index(drop=True)
assert len(valid_df) == 508, f"예상 validation 508개와 다릅니다: {len(valid_df)}"

def image_path(relative_path):
    path = Path(str(relative_path))
    return path if path.is_absolute() else DATA_ROOT / path

for frame in (gold_train_df, valid_df, test_df):
    missing_images = [str(path) for path in frame["path"].head(100) if not image_path(path).exists()]
    assert not missing_images, f"이미지 경로 확인 실패 예시: {missing_images[:3]}"

print("gold train:", len(gold_train_df), "/ valid:", len(valid_df), "/ dev:", len(dev_df), "/ test:", len(test_df))
print("\nvalid category\n", valid_df["category"].value_counts())
print("\ntest category\n", test_df["category"].value_counts())


## 4. 프롬프트와 공통 추론 함수


In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an expert visual multiple-choice question answering system. "
    "Inspect the entire image carefully. For quantity questions, count every relevant visible object exactly once. "
    "Answer with exactly one lowercase letter: a, b, c, or d. Do not explain."
)

def build_prompt(row):
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        "정답을 a, b, c, d 중 한 글자로만 출력하세요."
    )

def build_messages(row, image, answer=None):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_prompt(row)},
        ]},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(answer)}]})
    return messages

def apply_template(messages, add_generation_prompt):
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)

def evaluate_predictions(dataframe, title):
    assert "answer" in dataframe and "pred" in dataframe
    correct = int((dataframe["answer"] == dataframe["pred"]).sum())
    print(f"\n=== {title} ===")
    print(f"전체: {correct}/{len(dataframe)} = {correct/len(dataframe):.4f}")
    print(dataframe.groupby("category")["correct"].agg(["mean", "sum", "count"]))
    print("pred distribution:", dataframe["pred"].value_counts().sort_index().to_dict())
    return correct


## 5. Qwen3.5-9B BF16 로드


In [ ]:
from transformers import AutoProcessor
try:
    from transformers import Qwen3_5ForConditionalGeneration as Qwen35Model
except ImportError:
    from transformers import AutoModelForMultimodalLM as Qwen35Model

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

model = Qwen35Model.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.eval()
MODEL_DEVICE = next(model.parameters()).device
print("loaded on", MODEL_DEVICE, cuda_memory())

# 채점은 생성이 아니라 a/b/c/d 다음-token 확률을 직접 비교합니다.
dummy_messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
    {"role": "user", "content": [{"type": "text", "text": "Choose one: (a) A (b) B (c) C (d) D"}]},
]
dummy_prefix = apply_template(dummy_messages, add_generation_prompt=True)
dummy_ids = processor.tokenizer(dummy_prefix, add_special_tokens=False)["input_ids"]
LETTER_TOKEN_IDS = {}
for letter in LETTERS:
    extended_ids = processor.tokenizer(dummy_prefix + letter, add_special_tokens=False)["input_ids"]
    assert extended_ids[:len(dummy_ids)] == dummy_ids and len(extended_ids) > len(dummy_ids)
    suffix = extended_ids[len(dummy_ids):]
    LETTER_TOKEN_IDS[letter] = suffix[0]
assert len(set(LETTER_TOKEN_IDS.values())) == 4, LETTER_TOKEN_IDS
print("letter token ids:", LETTER_TOKEN_IDS)

def forward_last_logits(active_model, inputs):
    try:
        return active_model(**inputs, use_cache=False, logits_to_keep=1)
    except TypeError:
        return active_model(**inputs, use_cache=False)

def score_letters(active_model, dataframe, desc):
    old_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    active_model.eval()
    probabilities = []
    letter_tensor = torch.tensor([LETTER_TOKEN_IDS[x] for x in LETTERS], device=MODEL_DEVICE)
    with torch.inference_mode():
        for start in tqdm(range(0, len(dataframe), INFER_BATCH_SIZE), desc=desc, unit="batch"):
            chunk = dataframe.iloc[start:start + INFER_BATCH_SIZE]
            images, texts = [], []
            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = ImageOps.exif_transpose(opened).convert("RGB")
                images.append(image)
                texts.append(apply_template(build_messages(row, image), add_generation_prompt=True))
            inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(MODEL_DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = forward_last_logits(active_model, inputs)
            last_logits = outputs.logits[:, -1, :]
            probs = torch.softmax(last_logits.index_select(-1, letter_tensor).float(), dim=-1)
            probabilities.extend(probs.cpu().tolist())
            del inputs, outputs, last_logits, probs, images, texts
    processor.tokenizer.padding_side = old_side
    result = dataframe.reset_index(drop=True).copy()
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = [values[index] for values in probabilities]
    result["pred"] = [LETTERS[int(np.argmax(values))] for values in probabilities]
    result["pred_conf"] = [float(max(values)) for values in probabilities]
    if "answer" in result.columns:
        result["correct"] = result["pred"] == result["answer"]
    return result


## 6. Zero-shot 전체 validation


In [ ]:
if RUN_ZERO_SHOT_VALID:
    zero_valid = score_letters(model, valid_df, "Qwen3.5-9B zero-shot valid")
    zero_valid.to_csv(ZERO_VALID_PATH, index=False)
    zero_correct = evaluate_predictions(zero_valid, "Qwen3.5-9B zero-shot")
else:
    assert ZERO_VALID_PATH.exists(), f"RUN_ZERO_SHOT_VALID=False지만 결과가 없습니다: {ZERO_VALID_PATH}"
    zero_valid = pd.read_csv(ZERO_VALID_PATH)
    zero_correct = evaluate_predictions(zero_valid, "Qwen3.5-9B zero-shot cached")
print("memory:", cuda_memory())


## 7. Gold + 고신뢰 dev pseudo-label 학습셋


In [ ]:
VOTE_COLUMNS = ["answer1", "answer2", "answer3", "answer4", "answer5"]

def majority_vote(row):
    votes = [
        str(row[column]).strip().lower()
        for column in VOTE_COLUMNS
        if pd.notna(row[column]) and str(row[column]).strip().lower() in LETTERS
    ]
    if not votes:
        return pd.Series({"answer": None, "vote_count": 0, "vote_margin": 0})
    counts = Counter(votes)
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    answer, count = ordered[0]
    second = ordered[1][1] if len(ordered) > 1 else 0
    return pd.Series({"answer": answer, "vote_count": count, "vote_margin": count - second})

vote_result = dev_df.apply(majority_vote, axis=1)
dev_labeled = dev_df.drop(columns=VOTE_COLUMNS).copy()
dev_labeled[["answer", "vote_count", "vote_margin"]] = vote_result
dev_labeled = dev_labeled[
    (dev_labeled["vote_count"] >= MIN_DEV_VOTES) & dev_labeled["answer"].isin(LETTERS)
].copy()
dev_labeled["source"] = "dev_pseudo"
gold_train_df["source"] = "gold"
gold_train_df["vote_count"] = 99
gold_train_df["vote_margin"] = 99

# dev가 counting에 치우쳐 있으므로 category별 gold 수의 75%까지만 pseudo-label을 사용합니다.
selected_pseudo = []
for category, gold_group in gold_train_df.groupby("category"):
    candidates = dev_labeled[dev_labeled["category"] == category].copy()
    cap = int(math.ceil(len(gold_group) * PSEUDO_CAP_PER_CATEGORY))
    candidates = candidates.sort_values(
        ["vote_count", "vote_margin"], ascending=False, kind="stable"
    ).head(cap)
    selected_pseudo.append(candidates)
selected_pseudo_df = pd.concat(selected_pseudo, ignore_index=True)

finetune_df = pd.concat([gold_train_df, selected_pseudo_df], ignore_index=True, sort=False)
finetune_df = finetune_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("gold:", len(gold_train_df), "/ selected pseudo:", len(selected_pseudo_df), "/ total:", len(finetune_df))
print(finetune_df.groupby(["category", "source"]).size())


## 8. Vision + Language LoRA 부착


In [ ]:
from peft import LoraConfig, get_peft_model

LANGUAGE_LEAVES = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "in_proj_z", "out_proj",
}
VISION_LEAVES = {"qkv", "proj", "linear_fc1", "linear_fc2", "gate_proj", "up_proj", "down_proj"}

visual_targets, language_targets = [], []
for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    leaf = name.rsplit(".", 1)[-1]
    padded = f".{name}."
    if ".visual." in padded and leaf in VISION_LEAVES:
        visual_targets.append(name)
    elif ".language_model.layers." in padded and leaf in LANGUAGE_LEAVES:
        language_targets.append(name)

target_modules = visual_targets + language_targets
assert visual_targets, "Qwen3.5 vision LoRA 대상을 찾지 못했습니다. 모델 구조를 확인하세요."
assert language_targets, "Qwen3.5 language LoRA 대상을 찾지 못했습니다. 모델 구조를 확인하세요."
print("vision targets:", len(visual_targets), visual_targets[:10])
print("language targets:", len(language_targets), language_targets[:10])

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=target_modules,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False
model.print_trainable_parameters()
MODEL_DEVICE = next(model.parameters()).device
print("memory:", cuda_memory())


## 9. Dataset, collator, gradient smoke test


In [ ]:
class FullVQADataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(image_path(row["path"])) as opened:
            image = ImageOps.exif_transpose(opened).convert("RGB")
        has_horizontal_reference = any(
            term in str(row["question"]) for term in ["왼쪽", "오른쪽", "좌측", "우측"]
        )
        if AUGMENT_HORIZONTAL_FLIP and not has_horizontal_reference and random.random() < 0.5:
            image = ImageOps.mirror(image)
        return {"row": row, "image": image, "answer": str(row["answer"]).strip().lower()}

class LetterOnlyCollator:
    def __call__(self, items):
        images, texts, answers = [], [], []
        for item in items:
            row, image, answer = item["row"], item["image"], item["answer"]
            assert answer in LETTERS
            images.append(image)
            answers.append(answer)
            messages = build_messages(row, image, answer=answer)
            texts.append(apply_template(messages, add_generation_prompt=False))

        old_side = processor.tokenizer.padding_side
        processor.tokenizer.padding_side = "right"
        encoded = processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.full_like(encoded["input_ids"], -100)

        # assistant 답의 마지막 a/b/c/d 토큰 하나만 supervise합니다.
        for index, answer in enumerate(answers):
            sequence_length = int(encoded["attention_mask"][index].sum())
            token_id = LETTER_TOKEN_IDS[answer]
            candidates = torch.where(encoded["input_ids"][index, :sequence_length] == token_id)[0]
            candidates = candidates[candidates >= max(0, sequence_length - 32)]
            assert len(candidates) >= 1, (
                f"assistant answer token을 찾지 못했습니다: answer={answer}, "
                f"tail={processor.tokenizer.decode(encoded['input_ids'][index, max(0, sequence_length-32):sequence_length])}"
            )
            answer_position = int(candidates[-1])
            labels[index, answer_position] = token_id

        encoded["labels"] = labels
        processor.tokenizer.padding_side = old_side
        return encoded

train_dataset = FullVQADataset(finetune_df)
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=LetterOnlyCollator(),
)

sample_batch = next(iter(train_loader))
for row_index in range(sample_batch["labels"].shape[0]):
    supervised = sample_batch["labels"][row_index][sample_batch["labels"][row_index] != -100]
    assert len(supervised) == 1
    print("supervised sample", row_index, processor.tokenizer.decode(supervised.tolist()))

# 장시간 학습 전에 vision과 language LoRA 양쪽에 실제 gradient가 흐르는지 검증합니다.
model.train()
model.zero_grad(set_to_none=True)
smoke_batch = {
    key: value.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(value) else value
    for key, value in sample_batch.items()
}
with torch.autocast("cuda", dtype=torch.bfloat16):
    smoke_outputs = model(**smoke_batch, use_cache=False)
smoke_outputs.loss.backward()
visual_gradient_names, language_gradient_names = [], []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad or parameter.grad is None:
        continue
    if not torch.isfinite(parameter.grad).all() or float(parameter.grad.abs().max()) == 0.0:
        continue
    if "visual" in name:
        visual_gradient_names.append(name)
    if "language_model" in name:
        language_gradient_names.append(name)
assert visual_gradient_names, "vision LoRA에 유효한 gradient가 흐르지 않습니다."
assert language_gradient_names, "language LoRA에 유효한 gradient가 흐르지 않습니다."
print(
    "gradient smoke OK / loss", float(smoke_outputs.loss.detach()),
    "/ vision", len(visual_gradient_names), "/ language", len(language_gradient_names),
)
model.zero_grad(set_to_none=True)
del sample_batch, smoke_batch, smoke_outputs
clear_cuda()


## 10. 최대 3 epoch LoRA 학습과 epoch별 검증


In [ ]:
from transformers import get_cosine_schedule_with_warmup
from peft import get_peft_model_state_dict, set_peft_model_state_dict

assert 1 <= EPOCHS <= 3, "현재 프로필은 기존 실험 근거에 따라 최대 3 epoch까지만 허용합니다."
trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_updates = updates_per_epoch * EPOCHS
warmup_updates = int(total_updates * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_updates, total_updates)

history = []
global_update = 0
best_epoch = None
best_valid_correct = -1
best_valid_nll = float("inf")
best_adapter_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    progress = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Qwen3.5 epoch {epoch}")
    for step, batch in progress:
        batch = {
            key: value.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(value) else value
            for key, value in batch.items()
        }
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch, use_cache=False)
            raw_loss = outputs.loss
            loss = raw_loss / GRAD_ACCUM
        loss.backward()
        running_loss += float(raw_loss.detach())

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_update += 1

        if step % 25 == 0:
            progress.set_postfix(
                loss=f"{running_loss / step:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
                mem=f"{torch.cuda.max_memory_allocated()/1e9:.1f}G",
            )
        del batch, outputs, raw_loss, loss

    clear_cuda()
    epoch_valid = score_letters(model, valid_df, f"Qwen3.5 valid epoch {epoch}")
    valid_correct = int(epoch_valid["correct"].sum())
    probability_matrix = epoch_valid[[f"prob_{letter}" for letter in LETTERS]].to_numpy(dtype=np.float64)
    gold_indices = epoch_valid["answer"].map({letter: index for index, letter in enumerate(LETTERS)}).to_numpy()
    gold_probabilities = probability_matrix[np.arange(len(epoch_valid)), gold_indices]
    valid_nll = float(-np.log(np.clip(gold_probabilities, 1e-12, 1.0)).mean())

    epoch_record = {
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "updates": global_update,
        "valid_correct": valid_correct,
        "valid_accuracy": valid_correct / len(valid_df),
        "valid_nll": valid_nll,
    }
    for category, group in epoch_valid.groupby("category"):
        epoch_record[f"valid_{category}_correct"] = int(group["correct"].sum())
        epoch_record[f"valid_{category}_count"] = int(len(group))
    history.append(epoch_record)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    epoch_valid.to_csv(OUTPUT_ROOT / f"qwen35_valid_epoch{epoch}.csv", index=False)

    epoch_adapter_dir = OUTPUT_ROOT / f"adapter_epoch{epoch}"
    model.save_pretrained(epoch_adapter_dir, safe_serialization=True)

    # 정확도 우선, 동률이면 gold NLL이 낮은 epoch를 선택합니다.
    is_best = (
        valid_correct > best_valid_correct
        or (valid_correct == best_valid_correct and valid_nll < best_valid_nll)
    )
    if is_best:
        best_epoch = epoch
        best_valid_correct = valid_correct
        best_valid_nll = valid_nll
        best_adapter_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in get_peft_model_state_dict(model).items()
        }

    print(
        epoch_record,
        "/ best epoch", best_epoch,
        f"({best_valid_correct}/{len(valid_df)}, nll={best_valid_nll:.5f})",
        "/ memory", cuda_memory(),
    )
    model.train()
    clear_cuda()

assert best_adapter_state is not None and best_epoch is not None
load_result = set_peft_model_state_dict(model, best_adapter_state)
model.save_pretrained(BEST_ADAPTER_DIR, safe_serialization=True)
print(
    f"best epoch {best_epoch} adapter 복원 완료:", BEST_ADAPTER_DIR,
    f"valid={best_valid_correct}/{len(valid_df)}, nll={best_valid_nll:.5f}",
)
print("adapter load result:", load_result)

# 추론 전에 optimizer와 CPU best-state 복사본을 해제합니다.
del optimizer, scheduler, trainable_parameters, train_loader, train_dataset, best_adapter_state
clear_cuda()
print("optimizer 해제 후:", cuda_memory())


## 11. Best-epoch validation과 카테고리 router


In [ ]:
tuned_valid = score_letters(model, valid_df, "Qwen3.5-9B tuned valid")
tuned_valid.to_csv(TUNED_VALID_PATH, index=False)
tuned_correct = evaluate_predictions(tuned_valid, f"Qwen3.5-9B best epoch {best_epoch} LoRA")

comparison = zero_valid[["id", "answer", "category", "pred"]].rename(
    columns={"pred": "zero_pred"}
).merge(
    tuned_valid[["id", "pred"]].rename(columns={"pred": "tuned_pred"}),
    on="id",
    validate="one_to_one",
)

router_by_category = {}
router_rows = []
for category, group in comparison.groupby("category"):
    zero_category_correct = int((group["zero_pred"] == group["answer"]).sum())
    tuned_category_correct = int((group["tuned_pred"] == group["answer"]).sum())
    selected = "tuned" if tuned_category_correct > zero_category_correct else "zero"
    router_by_category[category] = selected
    router_rows.append({
        "category": category,
        "count": len(group),
        "zero_correct": zero_category_correct,
        "tuned_correct": tuned_category_correct,
        "selected": selected,
    })

comparison["pred"] = [
    row.tuned_pred if router_by_category[row.category] == "tuned" else row.zero_pred
    for row in comparison.itertuples()
]
comparison["correct"] = comparison["pred"] == comparison["answer"]
comparison.to_csv(ROUTED_VALID_PATH, index=False)
routed_correct = evaluate_predictions(comparison, "Qwen3.5 category-routed")

router_table = pd.DataFrame(router_rows).sort_values("category")
router_table.to_csv(OUTPUT_ROOT / "category_router.csv", index=False)
print("\ncategory router\n", router_table.to_string(index=False))

oracle_correct = int(
    ((comparison["zero_pred"] == comparison["answer"]) | (comparison["tuned_pred"] == comparison["answer"])).sum()
)
print("\nzero/tuned oracle:", f"{oracle_correct}/508 = {oracle_correct/508:.4f}")
print("현재 기존 최고 validation 참고: 469/508; 목표 0.96 validation: 488/508")
print("zero:", zero_correct, "/ tuned:", tuned_correct, "/ routed:", routed_correct)


## 12. Test 추론과 제출 3종 생성


In [ ]:
def write_submission(scored_test, path):
    submission = scored_test[["id", "pred"]].rename(columns={"pred": "answer"}).copy()
    assert len(submission) == len(test_df)
    assert submission["id"].tolist() == test_df["id"].tolist()
    assert submission["answer"].isin(LETTERS).all()
    submission.to_csv(path, index=False)
    print(path, submission["answer"].value_counts().sort_index().to_dict())
    return submission

if RUN_TEST:
    # adapter ON
    tuned_test = score_letters(model, test_df, "Qwen3.5-9B tuned test")
    tuned_test.to_csv(TUNED_TEST_PATH, index=False)
    tuned_submission = write_submission(tuned_test, SUB_TUNED_PATH)

    # adapter OFF: 동일 base model의 zero-shot test
    model.disable_adapter_layers()
    try:
        zero_test = score_letters(model, test_df, "Qwen3.5-9B zero-shot test")
    finally:
        model.enable_adapter_layers()
    zero_test.to_csv(ZERO_TEST_PATH, index=False)
    zero_submission = write_submission(zero_test, SUB_ZERO_PATH)

    routed_test = zero_test[["id", "category", "pred"]].rename(columns={"pred": "zero_pred"}).merge(
        tuned_test[["id", "pred"]].rename(columns={"pred": "tuned_pred"}),
        on="id",
        validate="one_to_one",
    )
    routed_test["pred"] = [
        row.tuned_pred if router_by_category[row.category] == "tuned" else row.zero_pred
        for row in routed_test.itertuples()
    ]
    routed_submission = write_submission(routed_test, SUB_ROUTED_PATH)
    routed_test.to_csv(OUTPUT_ROOT / "qwen35_routed_test.csv", index=False)

    print("\n제출 우선순위: validation 결과가 가장 높은 파일을 우선합니다.")
    print({
        "zero_valid": zero_correct,
        "tuned_valid": tuned_correct,
        "routed_valid": routed_correct,
        "recommended": str(
            SUB_ROUTED_PATH if routed_correct >= max(zero_correct, tuned_correct)
            else SUB_TUNED_PATH if tuned_correct >= zero_correct
            else SUB_ZERO_PATH
        ),
    })
else:
    print("RUN_TEST=False — validation까지만 완료했습니다.")


## 13. 실행 메타데이터와 Drive 백업


In [ ]:
metadata_payload = {
    "model_id": MODEL_ID,
    "image_size": IMAGE_SIZE,
    "seed": SEED,
    "epochs": EPOCHS,
    "train_rows": len(finetune_df),
    "gold_rows": len(gold_train_df),
    "pseudo_rows": len(selected_pseudo_df),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "zero_valid_correct": int(zero_correct),
    "tuned_valid_correct": int(tuned_correct),
    "routed_valid_correct": int(routed_correct),
    "router_by_category": router_by_category,
    "gpu": GPU_NAME,
    "gpu_vram_gib": GPU_VRAM_GIB,
}
with open(OUTPUT_ROOT / "run_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata_payload, file, ensure_ascii=False, indent=2)

if EXPORT_TO_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    drive_run = DRIVE_OUTPUT_ROOT / RUN_STAMP
    drive_run.mkdir(parents=True, exist_ok=True)
    artifacts = [
        ZERO_VALID_PATH, TUNED_VALID_PATH, ROUTED_VALID_PATH, HISTORY_PATH,
        OUTPUT_ROOT / "category_router.csv", OUTPUT_ROOT / "run_metadata.json",
        ZERO_TEST_PATH, TUNED_TEST_PATH, OUTPUT_ROOT / "qwen35_routed_test.csv",
        SUB_ZERO_PATH, SUB_TUNED_PATH, SUB_ROUTED_PATH,
    ]
    for artifact in artifacts:
        if Path(artifact).exists():
            shutil.copy2(artifact, drive_run / Path(artifact).name)
    if BEST_ADAPTER_DIR.exists():
        shutil.copytree(BEST_ADAPTER_DIR, drive_run / BEST_ADAPTER_DIR.name, dirs_exist_ok=True)
    print("Drive backup:", drive_run)

print("\n완료")
print("local output:", OUTPUT_ROOT)
print("memory:", cuda_memory())


## 14. Epoch2 test 확률과 epoch2/3 category router


In [ ]:
# Epoch3는 raw accuracy가 높고 epoch2는 NLL calibration이 더 좋으므로 둘 다 test 확률을 보존합니다.
from safetensors.torch import load_file
from peft import set_peft_model_state_dict

assert RUN_TEST and "tuned_test" in globals(), "먼저 12번 test 추론 셀까지 완료하세요."
epoch2_adapter_dir = OUTPUT_ROOT / "adapter_epoch2"
epoch2_adapter_file = epoch2_adapter_dir / "adapter_model.safetensors"
best_adapter_file = BEST_ADAPTER_DIR / "adapter_model.safetensors"
assert epoch2_adapter_file.exists(), f"epoch2 adapter가 없습니다: {epoch2_adapter_file}"
assert best_adapter_file.exists(), f"best adapter가 없습니다: {best_adapter_file}"

epoch2_state = load_file(str(epoch2_adapter_file), device="cpu")
epoch2_load_result = set_peft_model_state_dict(model, epoch2_state)
del epoch2_state
clear_cuda()
print("epoch2 adapter loaded:", epoch2_load_result)

epoch2_test = score_letters(model, test_df, "Qwen3.5-9B epoch2 test")
EPOCH2_TEST_PATH = OUTPUT_ROOT / "qwen35_epoch2_test.csv"
epoch2_test.to_csv(EPOCH2_TEST_PATH, index=False)

# 후속 사용을 위해 메모리상 모델은 다시 standalone best인 epoch3로 복원합니다.
best_state = load_file(str(best_adapter_file), device="cpu")
best_load_result = set_peft_model_state_dict(model, best_state)
del best_state
clear_cuda()
print("best adapter restored:", best_load_result)

epoch2_valid = pd.read_csv(OUTPUT_ROOT / "qwen35_valid_epoch2.csv")
epoch3_valid = tuned_valid.copy()
epoch_router = {}
epoch_router_rows = []
for category in sorted(valid_df["category"].unique()):
    valid2 = epoch2_valid[epoch2_valid["category"] == category]
    valid3 = epoch3_valid[epoch3_valid["category"] == category]
    correct2 = int(valid2["correct"].sum())
    correct3 = int(valid3["correct"].sum())
    # 동률이면 calibration이 더 좋은 epoch2를 선택합니다.
    selected_epoch = 3 if correct3 > correct2 else 2
    epoch_router[category] = selected_epoch
    epoch_router_rows.append({
        "category": category,
        "epoch2_correct": correct2,
        "epoch3_correct": correct3,
        "selected_epoch": selected_epoch,
    })

epoch_router_valid = epoch2_valid[["id", "answer", "category", "pred"]].rename(
    columns={"pred": "epoch2_pred"}
).merge(
    epoch3_valid[["id", "pred"]].rename(columns={"pred": "epoch3_pred"}),
    on="id",
    validate="one_to_one",
)
epoch_router_valid["pred"] = [
    row.epoch3_pred if epoch_router[row.category] == 3 else row.epoch2_pred
    for row in epoch_router_valid.itertuples()
]
epoch_router_valid["correct"] = epoch_router_valid["pred"] == epoch_router_valid["answer"]
EPOCH_ROUTER_VALID_PATH = OUTPUT_ROOT / "qwen35_epoch_router_valid.csv"
epoch_router_valid.to_csv(EPOCH_ROUTER_VALID_PATH, index=False)

epoch_router_test = epoch2_test[["id", "category", "pred"]].rename(
    columns={"pred": "epoch2_pred"}
).merge(
    tuned_test[["id", "pred"]].rename(columns={"pred": "epoch3_pred"}),
    on="id",
    validate="one_to_one",
)
epoch_router_test["pred"] = [
    row.epoch3_pred if epoch_router[row.category] == 3 else row.epoch2_pred
    for row in epoch_router_test.itertuples()
]
EPOCH_ROUTER_TEST_PATH = OUTPUT_ROOT / "qwen35_epoch_router_test.csv"
EPOCH_ROUTER_SUBMISSION_PATH = OUTPUT_ROOT / "submission_qwen35_epoch_router.csv"
epoch_router_test.to_csv(EPOCH_ROUTER_TEST_PATH, index=False)
write_submission(epoch_router_test, EPOCH_ROUTER_SUBMISSION_PATH)

epoch_router_correct = int(epoch_router_valid["correct"].sum())
print("\nEpoch2/3 category router:", f"{epoch_router_correct}/508 = {epoch_router_correct/508:.4f}")
print(pd.DataFrame(epoch_router_rows).to_string(index=False))

if EXPORT_TO_DRIVE and "drive_run" in globals():
    for artifact in [
        EPOCH2_TEST_PATH, EPOCH_ROUTER_VALID_PATH,
        EPOCH_ROUTER_TEST_PATH, EPOCH_ROUTER_SUBMISSION_PATH,
    ]:
        shutil.copy2(artifact, drive_run / artifact.name)
    print("epoch2/router artifacts copied:", drive_run)
